In [ ]:
from google import genai
from google.genai import types

import os
from dotenv import load_dotenv

# make a ".env" file and fill GEMINI_API_KEY=
# it is gitignored so no worries
load_dotenv() 
client = genai.Client() 

In [2]:
file_path = "/home/exvick/Desktop/Hackathon/data/input/120200009.0.pdf"

uploaded_file = client.files.upload(file=file_path)

In [3]:
json = {
  "sik_no": int, # the topmost number in the squares
  "voter_count": int, # брой на избирателите - точка 1)
  "additional_voter_count": int, # избиратели под чертата - точка 2)
  "registered_votes": int, # избирателите според положените подписи - точка 3)
  "paper_ballots": {
     "total": int, # тоталния брой хартиени бюлетини в секция - брой на получените бюлетини - точка А
     "unused_ballots": int, # точка 4а
     "registered_vote": int, # намерените бюлетините в кутията с хартии - точка 5)
     "invalid_out_of_the_box": int, # брой на недейстителните хартиени бюлетини за образци - точка 4б
     "invalid_in_the_box": int, # брой на недейстителните хартиени бюлетини в избирателната кутия - точка 6)
     "support_noone": int,
     "votes": [
        {
           "party_number": int, # номера на партията
           "votes": int, # разпределението за хартиените бюлетини общо без преференции - точка 8
           "preferences": [ # да се извлече от - точка 10
             {
                "candidate_number": int, # например 101
                "count": int # колко са гласували с тази преференция
             }
           ],
           "no_preferences": int # без преференция - точка 10
        },
        {
           "...": "..."
        }
     ],
     "total_valid_votes": int # общ брой на действителните гласове - точка 9
  },
  "machine_ballots": {
     "total_votes": int, # брой избиратели от машинно гласуване - точка 11 
     "support_noone": int,  # не подкрепям никого точка 12
     "total_valid_votes": int, # точка 14
     "votes": [
        {
           "party_number": int, # номер на партията - точка 13
           "votes": int, # действителни гласове - точка 13
           "preferences": [  # да се извлече от - точка 15
             {
               "candidate_number": int, # например 103
               "count": int # колко са гласували с тази преференция
             }
           ],
           "no_preferences": int # без преференция - точка 10
        },
        {
           "...": "..."
        }
     ]
  }
}

In [4]:
prompt = f"""
Ти си експерт по изборни одити. Разгледай този сканиран протокол, написан на ръка.
Намери и извлечи числата за следните конкретни полета от този JSON:
{json}
Ако дадено число е било задраскано, зачеркнато с черта или е напълно неразбираемо, върни -1. Върни въпросния JSON с пълните полета.
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash", 
    contents=[uploaded_file, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json", 
        temperature=0.0
    )
)

print("\n--- Extracted Protocol Data ---")
print(response.text)